## 1. Importação das bibliotecas

Este bloco importa as bibliotecas utilizadas ao longo do notebook:
- `pandas` (`pd`): leitura, organização, transformação e análise dos dados em tabelas.
- `numpy` (`np`): operações numéricas e tratamento de valores, incluindo a identificação de códigos que serão convertidos em valores ausentes.#texto aqui



In [2]:
import pandas as pd
import numpy as np


## 2. Leitura dos datasets dos dados de mortalidade de 2022 até 2025 e dados da Uf



In [3]:


arquivos = ['dados_mortalidade_2022.csv', 'dados_mortalidade_2023.csv', 'dados_mortalidade_2024.csv']

df  = pd.concat([pd.read_csv('dados/'+arq,encoding='utf-8',sep=',')for arq in arquivos])


df2 = pd.read_csv('dados/dados_mortalidade_2025.csv',encoding='utf-8',sep=';',decimal=',')

uf = pd.read_csv('dados/UF.csv')





C:\Users\Fabio\AppData\Local\Temp\ipykernel_23024\4074054733.py:3: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df  = pd.concat([pd.read_csv('dados/'+arq,encoding='utf-8',sep=',')for arq in arquivos])
C:\Users\Fabio\AppData\Local\Temp\ipykernel_23024\4074054733.py:3: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df  = pd.concat([pd.read_csv('dados/'+arq,encoding='utf-8',sep=',')for arq in arquivos])
C:\Users\Fabio\AppData\Local\Temp\ipykernel_23024\4074054733.py:3: DtypeWarning: Columns (53,66,77,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df  = pd.concat([pd.read_csv('dados/'+arq,encoding='utf-8',sep=',')for arq in arquivos])
C:\Users\Fabio\AppData\Local\Temp\ipykernel_23024\4074054733.py:6: DtypeWarning: Columns (56,57,58,59,65,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('dados/dado

# Extraindo o código da UF pelos primeiros dois digitos do código oficial do IBGE.

In [4]:
df.head()
uf['codUf'] = uf['Código Oficial IBGE (7 dígitos)'].astype(str).str[:2]
uf.head()


,codDici,Capital,Sigla UF,Código Oficial IBGE (7 dígitos),codUf
0,1,Aracaju,SE,2800308,28
1,2,Belém,PA,1501402,15
2,3,Belo Horizonte,MG,3106200,31
3,4,Boa Vista,RR,1400100,14
4,5,Campo Grande,MS,5002704,50


# Extração das Variaveis que serão utilizadas para a analise futura.

In [5]:
lista_variaveis = ['CAUSABAS','DTOBITO','CODMUNRES','SEXO','IDADE','TIPOBITO',
'RACACOR',
'ESC2010', 
'ESCFALAGR1',
'LOCOCOR',
'ASSISTMED']



df =  df[lista_variaveis]
df2 = df2[lista_variaveis]



df.head()

,CAUSABAS,DTOBITO,CODMUNRES,SEXO,IDADE,TIPOBITO,RACACOR,ESC2010,ESCFALAGR1,LOCOCOR,ASSISTMED
0,I219,21042022,354850,2,499,2,1.0,3.0,6.0,3,2.0
1,I693,22042022,354850,2,473,2,1.0,2.0,4.0,3,2.0
2,N133,22042022,354850,1,469,2,2.0,3.0,6.0,2,1.0
3,I739,23042022,354850,1,483,2,1.0,5.0,8.0,3,1.0
4,I259,20042022,355100,2,465,2,1.0,2.0,11.0,1,1.0


In [6]:
df['DTOBITO'].value_counts(dropna=False)
df['DTOBITO'] = df['DTOBITO'].astype(str).str.zfill(8)
df['DTOBITO'].astype(str).str.len().value_counts(dropna=False)



df2['DTOBITO'].value_counts(dropna=False)
df2['DTOBITO'] = df2['DTOBITO'].astype(str).str.zfill(8)
df2['DTOBITO'].astype(str).str.len().value_counts(dropna=False)

DTOBITO
8    937033
Name: count, dtype: int64

# Junção dados de 2022/2024 com os dados de mortalidade de 2025 e realizacao de filtros

Realização dos filtros necessários de acordo com a população que será analisada, idade de 18 até 44 anos, a causa da morte pelo CID 21 ifarto agudo do miocardio e
organizando a coluna data do obito para o formato data.


In [7]:
df3 = pd.concat([df,df2])

df3['codUf'] = df3['CODMUNRES'].astype(str).str[:2]
df3 = df3.loc[(df3['IDADE']>= 418.0) & (df3['IDADE']<= 444.0)]

df3 = df3.loc[(df3['CAUSABAS'].astype(str).str[:3] == "I21")]
df3['IDADE'] = df3['IDADE'].astype(str).str[1:3]

df3['DTOBITO'] = df3['DTOBITO'].astype(str)
df3['DTOBITO'] = pd.to_datetime(df3['DTOBITO'],format='%d%m%Y',errors='coerce')


df3['DTOBITO'].astype(str).str.len().value_counts(dropna=False)




DTOBITO
10    17783
Name: count, dtype: int64

# Criando a coluna Ano do Obito 
coluna ano do obito para realizar a analise futura por ano. 

In [8]:
df3['ANOOBITO'] = df3['DTOBITO'].dt.year
df3.head()

,CAUSABAS,DTOBITO,CODMUNRES,SEXO,IDADE,TIPOBITO,RACACOR,ESC2010,ESCFALAGR1,LOCOCOR,ASSISTMED,codUf,ANOOBITO
64,I219,2022-02-23,240810,1,43,2,1.0,3.0,12.0,3,2.0,24,2022
159,I219,2022-06-10,350450,1,41,2,1.0,5.0,8.0,3,1.0,35,2022
281,I219,2022-02-06,430600,1,41,2,1.0,1.0,10.0,4,2.0,43,2022
812,I219,2022-04-06,355030,1,30,2,4.0,5.0,8.0,3,NaN,35,2022
1161,I219,2022-01-20,521560,1,26,2,1.0,3.0,12.0,1,1.0,52,2022


In [9]:
#df3.groupby(['ANOOBITO','IDADE']).size().reset_index(name="Total_Obitos_Ano")

# Deletando Duplicatas.

In [10]:
df3.duplicated().sum()
df3 = df3.drop_duplicates()

In [11]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17780 entries, 64 to 936795
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   CAUSABAS    17780 non-null  object        
 1   DTOBITO     17780 non-null  datetime64[ns]
 2   CODMUNRES   17780 non-null  int64         
 3   SEXO        17780 non-null  int64         
 4   IDADE       17780 non-null  object        
 5   TIPOBITO    17780 non-null  int64         
 6   RACACOR     17601 non-null  float64       
 7   ESC2010     17132 non-null  float64       
 8   ESCFALAGR1  17132 non-null  float64       
 9   LOCOCOR     17780 non-null  int64         
 10  ASSISTMED   12037 non-null  float64       
 11  codUf       17780 non-null  object        
 12  ANOOBITO    17780 non-null  int32         
dtypes: datetime64[ns](1), float64(4), int32(1), int64(4), object(3)
memory usage: 1.8+ MB


# preenchimento de Nulos
criando função preenchimentoProporcional, no qual será utilizada o preenchimento de nulos de acordo com a proporção preenchida nos outros dados, fazendo uma proporção diretamente porporcional.

In [ ]:
def preenchimentoProporcional(df,nomeColuna):
    proporcao = df[nomeColuna].value_counts(normalize=True)
    opcoes = proporcao.index.tolist()
    probabilidade = proporcao.values.tolist()
    total_nulos = df[nomeColuna].isna().sum()
    # Passo 4: Gerar uma lista de 1, 2 e 9 seguindo rigorosamente essas probabilidades
    valores_gerados = np.random.choice(opcoes, size=total_nulos, p=probabilidade)
    df.loc[df[nomeColuna].isna(), nomeColuna] = valores_gerados
    
    

def preenchimento0e9(df,nomeColuna):
    proporcao = df[nomeColuna].value_counts(normalize=True)
    opcoes = proporcao.index.tolist()
    probabilidade = proporcao.values.tolist()
    total_nulos = df[nomeColuna].isna().sum()
    # Passo 4: Gerar uma lista de 1, 2 e 9 seguindo rigorosamente essas probabilidades
    valores_gerados = np.random.choice(opcoes, size=total_nulos, p=probabilidade)
    df.loc[df[nomeColuna].isna(), nomeColuna] = valores_gerados

In [13]:
##PREECHIMENTO DE NULOS PROPORCIONAL

preenchimentoProporcional(df3,'ASSISTMED')
preenchimentoProporcional(df3,'RACACOR')
preenchimentoProporcional(df3,'ESC2010')
preenchimentoProporcional(df3,'ESCFALAGR1')



In [29]:
#verificação se todos os dados possuem nulos.
#ESC2010 e ESCFALAGR1
df3.loc[df3['ESCFALAGR1'] == 9].count()


CAUSABAS      1446
DTOBITO       1446
CODMUNRES     1446
SEXO          1446
IDADE         1446
TIPOBITO      1446
RACACOR       1446
ESC2010       1446
ESCFALAGR1    1446
LOCOCOR       1446
ASSISTMED     1446
codUf         1446
ANOOBITO      1446
dtype: int64

# Colocando nome dos Municipios na Base de dados

In [15]:
dfFinal = pd.merge(df3, uf, on='codUf',how='left')



dfFinal = dfFinal.drop(columns=(['CODMUNRES','codDici','Código Oficial IBGE (7 dígitos)']))

dfFinal

,CAUSABAS,DTOBITO,SEXO,IDADE,TIPOBITO,RACACOR,ESC2010,ESCFALAGR1,LOCOCOR,ASSISTMED,codUf,ANOOBITO,Capital,Sigla UF
0,I219,2022-02-23,1,43,2,1.0,3.0,12.0,3,2.0,24,2022,Natal,RN
1,I219,2022-06-10,1,41,2,1.0,5.0,8.0,3,1.0,35,2022,São Paulo,SP
2,I219,2022-02-06,1,41,2,1.0,1.0,10.0,4,2.0,43,2022,Porto Alegre,RS
3,I219,2022-04-06,1,30,2,4.0,5.0,8.0,3,1.0,35,2022,São Paulo,SP
4,I219,2022-01-20,1,26,2,1.0,3.0,12.0,1,1.0,52,2022,Goiânia,GO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17775,I219,2025-07-09,2,44,2,4.0,3.0,6.0,3,9.0,14,2025,Boa Vista,RR
17776,I219,2025-07-14,1,33,2,1.0,2.0,11.0,1,9.0,35,2025,São Paulo,SP
17777,I219,2025-07-11,1,43,2,1.0,1.0,2.0,3,1.0,17,2025,Palmas,TO
17778,I219,2025-07-18,2,42,2,1.0,2.0,4.0,1,1.0,35,2025,São Paulo,SP


In [16]:
dfFinal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17780 entries, 0 to 17779
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   CAUSABAS    17780 non-null  object        
 1   DTOBITO     17780 non-null  datetime64[ns]
 2   SEXO        17780 non-null  int64         
 3   IDADE       17780 non-null  object        
 4   TIPOBITO    17780 non-null  int64         
 5   RACACOR     17780 non-null  float64       
 6   ESC2010     17780 non-null  float64       
 7   ESCFALAGR1  17780 non-null  float64       
 8   LOCOCOR     17780 non-null  int64         
 9   ASSISTMED   17780 non-null  float64       
 10  codUf       17780 non-null  object        
 11  ANOOBITO    17780 non-null  int32         
 12  Capital     17780 non-null  object        
 13  Sigla UF    17780 non-null  object        
dtypes: datetime64[ns](1), float64(4), int32(1), int64(3), object(5)
memory usage: 1.8+ MB


### Renomear colunas 
'CAUSABAS', 
'DTOBITO', 
'SEXO',
'IDADE',
'TIPOBITO',
'RACACOR',
'ESC2010',
'ESCFALAGR1',
'LOCOCOR',
'ASSISTMED',
'codUf',
'ANOOBITO',
'Capital',
'Sigla UF'

# Exportando para csv os dados para serem utilizados para a analise dos dados.

In [17]:
#dfFinal.to_csv('dados/dados_SIM_Para_Analise.csv',index=False)